# ============================================================
# STEP 1 — Overview: Script Description
# ============================================================
# This script automates updating ACH (Air Changes per Hour) values 
# in ESP-r model configuration files for multiple buildings.
#
# Workflow:
# 1. Load EPC dataset containing BUILDING_REFERENCE_NUMBER and ACH_NATURAL
# 2. Loop through each building folder in baseline_models
# 3. Identify the correct model subfolder (e.g. Detached_2F, Flat, etc.)
# 4. Locate the cfg/ACH.txt file
# 5. Replace all instances of the string "ACH" with the building-specific ACH_NATURAL value
# 6. Save the updated file
# 7. Skip buildings where:
#    - ACH value is missing
#    - cfg/ACH.txt does not exist
#
# Notes:
# - The script is non-destructive except for replacing "ACH" tokens
# - It prints progress for debugging and tracking
# - Designed for use in Jupyter Notebook or VSCode (Python)
# ============================================================

In [ ]:
# ============================================================
# STEP 2 — Import Libraries & Define Paths
# ============================================================

import os
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
baseline_models_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

print("Paths set successfully.")

Paths set successfully.


In [14]:
# ============================================================
# STEP 3 — Load EPC Dataset
# ============================================================

# Load CSV
df = pd.read_csv(epc_file)

# Ensure correct data types
df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary: {building_id: ACH_value}
ach_lookup = dict(zip(df["BUILDING_REFERENCE_NUMBER"], df["ACH_NATURAL"]))

print(f"Loaded EPC data for {len(ach_lookup)} buildings.")

Loaded EPC data for 117 buildings.


In [ ]:
# ============================================================
# STEP 4 — Define Valid Model Folder Names
# ============================================================

model_folder_names = [
    "SemiDetached_2F",
    "Detached_2F"
]

print("Model folder names defined.")

Model folder names defined.


In [16]:
# ============================================================
# STEP 5 — Process Each Building Folder
# ============================================================

processed = 0
skipped_no_ach = 0
skipped_no_file = 0

# Loop through each building directory
for building_id in os.listdir(baseline_models_path):
    
    building_path = os.path.join(baseline_models_path, building_id)
    
    if not os.path.isdir(building_path):
        continue
    
    # Check if building exists in EPC dataset
    if building_id not in ach_lookup:
        print(f"[SKIP] {building_id}: Not found in EPC dataset")
        skipped_no_ach += 1
        continue
    
    ach_value = ach_lookup[building_id]
    
    # Skip if ACH value is missing or NaN
    if pd.isna(ach_value):
        print(f"[SKIP] {building_id}: ACH_NATURAL is NaN")
        skipped_no_ach += 1
        continue
    
    # Convert to string for replacement
    ach_str = str(round(float(ach_value), 3))
    
    # Find model folder
    model_folder_path = None
    for folder_name in model_folder_names:
        potential_path = os.path.join(building_path, folder_name)
        if os.path.isdir(potential_path):
            model_folder_path = potential_path
            break
    
    if model_folder_path is None:
        print(f"[SKIP] {building_id}: No model folder found")
        skipped_no_file += 1
        continue
    
    # Path to ACH.txt
    ach_file_path = os.path.join(model_folder_path, "cfg", "ACH.txt")
    
    if not os.path.isfile(ach_file_path):
        print(f"[SKIP] {building_id}: ACH.txt not found")
        skipped_no_file += 1
        continue
    
    # Read file
    with open(ach_file_path, "r") as file:
        content = file.read()
    
    # Replace "ACH" with actual value
    updated_content = content.replace("ACH", ach_str)
    
    # Write updated file
    with open(ach_file_path, "w") as file:
        file.write(updated_content)
    
    print(f"[UPDATED] {building_id}: ACH set to {ach_str}")
    processed += 1

print("\n============================================================")
print(f"Processing complete.")
print(f"Updated files: {processed}")
print(f"Skipped (no ACH): {skipped_no_ach}")
print(f"Skipped (no file/folder): {skipped_no_file}")
print("============================================================")

[SKIP] 1001664924: ACH.txt not found
[SKIP] 1001829085: ACH.txt not found
[SKIP] 1001063639: ACH.txt not found
[SKIP] 1001829071: ACH.txt not found
[SKIP] 1234761002: ACH.txt not found
[SKIP] 1002539407: ACH.txt not found
[SKIP] 1001063637: ACH.txt not found
[SKIP] 1001664941: ACH.txt not found
[SKIP] 1001991633: ACH.txt not found
[SKIP] 1235057414: ACH.txt not found
[SKIP] 1001829079: ACH.txt not found
[SKIP] 1001664922: ACH.txt not found
[SKIP] 1234761003: ACH.txt not found
[SKIP] 1001664925: ACH.txt not found
[SKIP] 1000238907: ACH.txt not found
[SKIP] 1234761004: ACH.txt not found
[SKIP] 1002313096: ACH.txt not found
[SKIP] 1001870878: ACH.txt not found
[SKIP] 1001664940: ACH.txt not found
[SKIP] 1234982990: ACH.txt not found
[SKIP] 1234806523: ACH.txt not found
[SKIP] 1001870876: ACH.txt not found
[SKIP] 1001870882: ACH.txt not found
[SKIP] 1002143357: ACH.txt not found
[SKIP] 1001951902: ACH.txt not found
[SKIP] 1234621926: ACH.txt not found
[SKIP] 1234647955: No model folder fou